# H2-1 extension, step 3: household x week panel (weeks 33-101)

**Carried over from the previous step**: the tier-fixing window was corrected to weeks 17-32 (to match the team's stable-window rule of week 17 onward).

**What this step does**
1. Attach weekly spend to the (household_key, WEEK_NO) skeleton
2. Attach whether each of TypeA/B/C is active, per household x week (separate dummies, not mutually exclusive)
3. Attach the tier labels
4. **Answer the question left open in the last interpretation: "did households receive two or more types in the same week?"**
5. Preliminary comparison for reference before the main regression (plain means, before fixed effects)

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

from config import (DATA_DIR, OUT_DIR, INTERMEDIATE_DIR, FIG_DIR, SEED, TIER_MIN_WEEK, TIER_MAX_WEEK,
                    CAMP_MIN_WEEK, CAMP_MAX_WEEK, MID_WEEK, CELL_SIZE_THRESH, TIER_EN,
                    load_tiers, load_panel, load_camp_full)

TIER_MAX_WEEK = 32
CAMP_MAX_WEEK = 101


## (Rebuild) steps 1-2, recomputed on the week-17 basis

To pick up the previous outputs unchanged, steps 1-2 are run again on the week-17 basis.

In [2]:
tx = pd.read_csv(DATA_DIR / "transaction_data.csv", usecols=["household_key", "DAY", "WEEK_NO", "SALES_VALUE"])
tier_df = load_tiers()          # written by h2_stage1_2
camp_full = load_camp_full()    # written by h2_stage1_2
all_households = tier_df.index.values
recipients = set(tier_df.index[tier_df["will_receive"] == 1])
never_recipients = set(tier_df.index[tier_df["will_receive"] == 0])
n_weeks_tier = TIER_MAX_WEEK - TIER_MIN_WEEK + 1
print(f"[Re-check] Tier-fixing window: weeks {TIER_MIN_WEEK}-{TIER_MAX_WEEK} ({n_weeks_tier} weeks)")
print(f"Households with zero pre-period transactions (floor-effect group): {tier_df['was_zero_pre'].sum()}")
print(f"Three-tier distribution: {tier_df['tier3'].value_counts().to_dict()}")

[Re-check] Tier-fixing window: weeks 17-32 (16 weeks)
Households with zero pre-period transactions (floor-effect group): 142
Three-tier distribution: {'low': 834, 'mid': 834, 'high': 832}


In [3]:
print("Steps 1-2 were loaded from outputs/intermediate (h2_stage1_2)")

Steps 1-2 were loaded from outputs/intermediate (h2_stage1_2)


## Step 3: household x week panel (weeks 33-101)

### 3-1. Panel skeleton + weekly spend

In [4]:
weeks_camp = list(range(CAMP_MIN_WEEK, CAMP_MAX_WEEK + 1))
n_weeks_camp = len(weeks_camp)
print(f"Campaign observation window: weeks {CAMP_MIN_WEEK}-{CAMP_MAX_WEEK} ({n_weeks_camp} weeks)")
print(f"Expected panel rows: {len(all_households):,} households x {n_weeks_camp} weeks = {len(all_households)*n_weeks_camp:,}")

panel_index = pd.MultiIndex.from_product([all_households, weeks_camp], names=["household_key", "WEEK_NO"])
panel = pd.DataFrame(index=panel_index).reset_index()

weekly_spend = (
    tx[(tx["WEEK_NO"] >= CAMP_MIN_WEEK) & (tx["WEEK_NO"] <= CAMP_MAX_WEEK)]
    .groupby(["household_key", "WEEK_NO"])["SALES_VALUE"].sum()
    .rename("spend")
)
panel = panel.merge(weekly_spend, on=["household_key", "WEEK_NO"], how="left")
panel["spend"] = panel["spend"].fillna(0)

print(f"\nActual panel rows: {len(panel):,}")
print(f"Share of rows with zero weekly spend: {(panel['spend']==0).mean()*100:.1f}% (household-weeks with no visit at all)")


Campaign observation window: weeks 33-101 (69 weeks)
Expected panel rows: 2,500 households x 69 weeks = 172,500

Actual panel rows: 172,500
Share of rows with zero weekly spend: 46.9% (household-weeks with no visit at all)


### 3-2. Active flag per campaign type (vectorized)

In [5]:
def build_active_weeks(camp_df, type_name):
    sub = camp_df[camp_df["DESCRIPTION_desc"] == type_name][["household_key", "START_WEEK", "END_WEEK"]].copy()
    sub["START_WEEK"] = sub["START_WEEK"].clip(lower=CAMP_MIN_WEEK)
    sub["END_WEEK"] = sub["END_WEEK"].clip(upper=CAMP_MAX_WEEK)
    sub["WEEK_NO"] = sub.apply(lambda r: list(range(int(r["START_WEEK"]), int(r["END_WEEK"]) + 1)), axis=1)
    sub = sub.explode("WEEK_NO")[["household_key", "WEEK_NO"]].drop_duplicates()
    sub["WEEK_NO"] = sub["WEEK_NO"].astype(int)
    sub[f"active_{type_name}"] = 1
    return sub

for t in ["TypeA", "TypeB", "TypeC"]:
    active = build_active_weeks(camp_full, t)
    panel = panel.merge(active, on=["household_key", "WEEK_NO"], how="left")
    panel[f"active_{t}"] = panel[f"active_{t}"].fillna(0).astype(int)
    n_active_pairs = int(panel[f"active_{t}"].sum())
    n_active_hh = panel.loc[panel[f"active_{t}"]==1, "household_key"].nunique()
    print(f"{t}: {n_active_pairs:,} active household-week pairs / {n_active_hh:,} households involved")


TypeA: 28,654 active household-week pairs / 1,513 households involved
TypeB: 13,260 active household-week pairs / 1,023 households involved
TypeC: 5,788 active household-week pairs / 397 households involved


### 3-3. Attach tier labels + check the panel

In [6]:
panel = panel.merge(tier_df[["tier3", "tier5", "was_zero_pre"]], left_on="household_key", right_index=True, how="left")

print(f"Final panel shape: {panel.shape}")
print()
print(panel.head(8).to_string(index=False))


Final panel shape: (172500, 9)

 household_key  WEEK_NO  spend  active_TypeA  active_TypeB  active_TypeC tier3       tier5  was_zero_pre
          2375       33   0.00             0             0             0   low q1 (lowest)          True
          2375       34   0.00             0             0             0   low q1 (lowest)          True
          2375       35  15.81             0             0             0   low q1 (lowest)          True
          2375       36   0.00             0             0             0   low q1 (lowest)          True
          2375       37  39.95             0             0             0   low q1 (lowest)          True
          2375       38   0.00             0             0             0   low q1 (lowest)          True
          2375       39   0.00             0             0             0   low q1 (lowest)          True
          2375       40   0.00             0             0             0   low q1 (lowest)          True


In [7]:
# --- the panel every later stage reads (household level, git-ignored) and its summary
panel.to_csv(INTERMEDIATE_DIR / "panel.csv", index=False)
pd.DataFrame([{
    "n_rows": len(panel), "n_households": panel["household_key"].nunique(), "n_weeks": panel["WEEK_NO"].nunique(),
    "first_week": panel["WEEK_NO"].min(), "last_week": panel["WEEK_NO"].max(),
    "zero_spend_rows_pct": (panel["spend"] == 0).mean() * 100,
}]).to_csv(OUT_DIR / "panel_summary.csv", index=False)
pd.DataFrame([{"type": t, "n_active_household_weeks": int(panel[f"active_{t}"].sum()),
               "n_active_households": panel.loc[panel[f"active_{t}"] == 1, "household_key"].nunique()}
              for t in ["TypeA", "TypeB", "TypeC"]]).to_csv(OUT_DIR / "exposure_by_type.csv", index=False)

### 3-4. [Key check] Are there households that received two or more types in the same week?

The last interpretation noted that "1,008 households received two or more types, but we need to check whether simultaneously or at different times".

In [8]:
panel["n_types_active_this_week"] = panel[["active_TypeA", "active_TypeB", "active_TypeC"]].sum(axis=1)
same_week_overlap = panel[panel["n_types_active_this_week"] >= 2]
print(f"Household-weeks with two or more types active in the same week: {len(same_week_overlap):,}")
print(f"Unique households involved: {same_week_overlap['household_key'].nunique():,}")

combo = same_week_overlap.apply(
    lambda r: "+".join([t for t in ["TypeA","TypeB","TypeC"] if r[f"active_{t}"]==1]), axis=1
)
print()
print("[Counts per simultaneous-active combination]")
print(combo.value_counts())

n_multi_type_total = camp_full.groupby("household_key")["DESCRIPTION_desc"].nunique().gt(1).sum()
n_same_week_hh = same_week_overlap['household_key'].nunique()
print(f"\nHouseholds that received two or more types (including at different times): {n_multi_type_total:,}")
print(f"Of these, households that overlapped in the same week: {n_same_week_hh:,} ({n_same_week_hh/n_multi_type_total*100:.1f}%)")


Household-weeks with two or more types active in the same week: 7,110
Unique households involved: 869

[Counts per simultaneous-active combination]
TypeA+TypeB          3530
TypeA+TypeC          1585
TypeB+TypeC          1434
TypeA+TypeB+TypeC     561
Name: count, dtype: int64

Households that received two or more types (including at different times): 1,008
Of these, households that overlapped in the same week: 869 (86.2%)


In [9]:
pd.DataFrame({"combination": combo.value_counts().index, "n_household_weeks": combo.value_counts().values}) \
    .to_csv(OUT_DIR / "overlap_combinations.csv", index=False)
pd.DataFrame([{
    "n_overlap_household_weeks": len(same_week_overlap), "pct_of_panel_rows": len(same_week_overlap) / len(panel) * 100,
    "n_overlap_households": int(n_same_week_hh), "n_multi_type_households": int(n_multi_type_total),
    "pct_overlap_among_multi_type": n_same_week_hh / n_multi_type_total * 100,
}]).to_csv(OUT_DIR / "overlap_summary.csv", index=False)

### 3-5. Preliminary check: mean spend in campaign-active vs campaign-inactive weeks (before fixed effects, for reference)

In [10]:
panel["any_active"] = (panel["n_types_active_this_week"] >= 1).astype(int)
prelim = panel.groupby(["tier3", "any_active"], observed=True)["spend"].mean().unstack()
prelim.columns = ["Mean spend, campaign-inactive weeks", "Mean spend, campaign-active weeks"]
prelim["Difference"] = prelim["Mean spend, campaign-active weeks"] - prelim["Mean spend, campaign-inactive weeks"]
prelim["Change (%)"] = (prelim["Difference"] / prelim["Mean spend, campaign-inactive weeks"] * 100)
print(prelim.round(2))
print()
print("Note: plain mean comparison without household and week fixed effects; it may differ from the step 6 main regression")


       Mean spend, campaign-inactive weeks  Mean spend, campaign-active weeks  \
tier3                                                                           
low                                  11.54                              39.49   
mid                                  22.48                              36.08   
high                                 61.68                              75.61   

       Difference  Change (%)  
tier3                          
low         27.95      242.27  
mid         13.60       60.50  
high        13.93       22.58  

Note: plain mean comparison without household and week fixed effects; it may differ from the step 6 main regression


In [11]:
# --- raw (pooled, no fixed effects) comparison of active vs inactive weeks, by tier
_long = panel.groupby(["tier3", "any_active"], observed=True)["spend"].agg(n_household_weeks="size", mean_spend="mean").reset_index()
_long["tier"] = _long["tier3"].astype(str).map(TIER_EN)
_long[["tier", "any_active", "n_household_weeks", "mean_spend"]].to_csv(OUT_DIR / "spend_by_tier_active.csv", index=False)
_w = _long.pivot(index="tier", columns="any_active", values="mean_spend").loc[["low", "mid", "high"]]
_n = _long.pivot(index="tier", columns="any_active", values="n_household_weeks").loc[["low", "mid", "high"]]
pd.DataFrame({
    "mean_spend_inactive_weeks": _w[0], "mean_spend_active_weeks": _w[1],
    "n_inactive_household_weeks": _n[0], "n_active_household_weeks": _n[1],
    "difference": _w[1] - _w[0], "pct_change": (_w[1] - _w[0]) / _w[0] * 100,
}).reset_index().to_csv(OUT_DIR / "raw_active_vs_inactive.csv", index=False)

**Observation for reference**: the change for the low tier (+242%) looks much larger than for the medium (+60%) and high (+23%) tiers,
but at this point two competing interpretations are possible and **cannot be told apart yet**.

1. Low spenders really respond more strongly to the campaigns
2. The **floor effect** pointed out in the last interpretation (the 142 households with zero transactions in the pre-period are concentrated in the low tier,
   so even a small amount of spending looks like an explosion in percent terms), plus the illusion of a plain average that does not yet control for household and week fixed effects

-> This table is never used for conclusions; it is interpreted only by comparing it with the step 6 main regression (with fixed effects).

## Next step
A multicollinearity issue was newly found in step 3, so before moving to step 4 (type x tier cell-size table)
it is better to **decide first how to handle this simultaneous-exposure problem**.

In [12]:

import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ============================================================
# VIF check 1: the 3 raw type-active dummies (the most basic check)
# ============================================================
X_raw = panel[["active_TypeA", "active_TypeB", "active_TypeC"]].astype(float)

vif_raw = pd.DataFrame({
    "variable": X_raw.columns,
    "VIF": [variance_inflation_factor(X_raw.values, i) for i in range(X_raw.shape[1])]
})
print("[VIF check 1] raw type-active dummies")
print(vif_raw)
print()

# ============================================================
# VIF check 2: the 9 type x tier interaction dummies that enter the step 6 regression
#            (this is the one that matters: collinearity among the interaction terms, not the raw dummies)
# ============================================================
interaction_cols = []
for t in ["TypeA", "TypeB", "TypeC"]:
    for tier in ["low", "mid", "high"]:
        col = f"{t}_x_{tier}"
        panel[col] = ((panel[f"active_{t}"] == 1) & (panel["tier3"] == tier)).astype(int)
        interaction_cols.append(col)

X_inter = panel[interaction_cols].astype(float)

vif_inter = pd.DataFrame({
    "variable": X_inter.columns,
    "VIF": [variance_inflation_factor(X_inter.values, i) for i in range(X_inter.shape[1])]
})
print("[VIF check 2] type x tier interaction dummies (variables entering the main regression)")
print(vif_inter.sort_values("VIF", ascending=False, kind="stable"))
print()

# ============================================================
# For reference: correlations among the raw type dummies (helpful to read together with the VIF)
# ============================================================
corr_raw = X_raw.corr()
print("[Reference] Correlations among the type-active dummies")
print(corr_raw.round(3))

# ============================================================
# Decision thresholds (footnote for reference)
# ============================================================
print("""
Rules of thumb:
  VIF < 5   : no problem, proceed as is
  5 <= VIF < 10 : caution, state that the standard error of that variable is wide when interpreting
  VIF >= 10 : severe, action needed such as (b) excluding simultaneous exposure or (a) combination dummies
""")

[VIF check 1] raw type-active dummies
       variable       VIF
0  active_TypeA  1.062637
1  active_TypeB  1.089636
2  active_TypeC  1.071389



[VIF check 2] type x tier interaction dummies (variables entering the main regression)
       variable       VIF
5  TypeB_x_high  1.117451
8  TypeC_x_high  1.098625
2  TypeA_x_high  1.079771
0   TypeA_x_low  1.044315
4   TypeB_x_mid  1.039543
3   TypeB_x_low  1.037779
1   TypeA_x_mid  1.036125
7   TypeC_x_mid  1.023822
6   TypeC_x_low  1.016183

[Reference] Correlations among the type-active dummies
              active_TypeA  active_TypeB  active_TypeC
active_TypeA         1.000         0.110         0.102
active_TypeB         0.110         1.000         0.187
active_TypeC         0.102         0.187         1.000

Rules of thumb:
  VIF < 5   : no problem, proceed as is
  5 <= VIF < 10 : caution, state that the standard error of that variable is wide when interpreting
  VIF >= 10 : severe, action needed such as (b) excluding simultaneous exposure or (a) combination dummies



In [13]:
_tier_en = lambda s: s  # internal tier labels are already the English names written to the CSVs
pd.concat([vif_raw.assign(group="type_dummies"), vif_inter.assign(group="type_x_tier")]) \
    .rename(columns={"VIF": "vif"}).assign(variable=lambda d: d["variable"].map(_tier_en)) \
    [["group", "variable", "vif"]].to_csv(OUT_DIR / "vif.csv", index=False)
corr_raw.rename_axis("variable").reset_index().to_csv(OUT_DIR / "type_correlation.csv", index=False)

# --- why week 102 is outside the panel: coverage of each week
wc = tx.groupby("WEEK_NO").agg(n_days=("DAY", "nunique"), first_day=("DAY", "min"), last_day=("DAY", "max"),
                               n_rows=("SALES_VALUE", "size"), n_households=("household_key", "nunique"),
                               sales_value=("SALES_VALUE", "sum")).reset_index().rename(columns={"WEEK_NO": "week_no"})
wc["rows_per_day"] = wc["n_rows"] / wc["n_days"]
wc["sales_per_day"] = wc["sales_value"] / wc["n_days"]
wc["in_panel"] = wc["week_no"].between(CAMP_MIN_WEEK, CAMP_MAX_WEEK)
wc.to_csv(OUT_DIR / "week_coverage.csv", index=False)
_last = wc["week_no"].max()
_ref = wc[wc["week_no"].between(CAMP_MAX_WEEK - 8, CAMP_MAX_WEEK)]
_pan = wc[wc["in_panel"]]
_rows = []
for _m in ["n_days", "n_rows", "n_households", "sales_value", "rows_per_day", "sales_per_day"]:
    _v = wc.loc[wc["week_no"] == _last, _m].iloc[0]
    _rows.append({"metric": _m, "last_week": _v, "mean_weeks_93_101": _ref[_m].mean(), "mean_panel_weeks": _pan[_m].mean(),
                  "min_panel_weeks": _pan[_m].min(), "ratio_to_mean_93_101": _v / _ref[_m].mean()})
pd.DataFrame(_rows).to_csv(OUT_DIR / "week_last_vs_reference.csv", index=False)
_desc = pd.read_csv(OUT_DIR / "campaign_timeline.csv")
pd.DataFrame([{
    "last_week": _last, "last_data_day": tx["DAY"].max(), "days_in_last_week": int(wc.loc[wc["week_no"] == _last, "n_days"].iloc[0]),
    "weeks_with_fewer_than_7_days": ",".join(str(w) for w in wc.loc[wc["n_days"] < 7, "week_no"]),
    "n_campaigns_end_day_after_last_data_day": int((_desc["end_day"] > tx["DAY"].max()).sum()),
    "n_campaigns_end_week_equals_last_week": int((_desc["end_week"] == _last).sum()),
}]).to_csv(OUT_DIR / "week_last_facts.csv", index=False)